# Apache Spark 架构原理

> **高频考点** | Senior Data Engineer 面试准备

## 目录
1. Driver / Executor / Task / Stage 关系
2. DAG Scheduler & Task Scheduler
3. 宽依赖 vs 窄依赖 & Shuffle
4. Catalyst Optimizer & Physical Plan
5. Tungsten 内存管理 & Code Generation
6. 练习题

---

## 1. Driver / Executor / Task / Stage 关系

### 核心概念

Spark 采用 **主从架构 (Master-Worker)**：

- **Driver**：运行用户代码的 JVM 进程，负责协调整个 Spark 应用
- **Executor**：在 Worker 节点上运行的 JVM 进程，负责执行 Task 并存储缓存数据
- **Task**：最小的执行单元，对应一个数据分区的计算
- **Stage**：一组可以 pipeline 执行的 Task 集合（以 Shuffle 为边界）
- **Job**：一个 Action 触发的完整计算

```
┌─────────────────────────────────────────────────────────────────┐
│                        Spark Application                        │
│                                                                 │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │                       DRIVER                            │   │
│  │                                                         │   │
│  │  SparkContext ──► DAG Scheduler ──► Task Scheduler      │   │
│  │       │                                    │            │   │
│  │  (Job = Action)              (分发 Task 到 Executor)     │   │
│  └────────────────────────────────────────────────────────┘    │
│                              │                                  │
│              ┌───────────────┼───────────────┐                  │
│              ▼               ▼               ▼                  │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐          │
│  │  Executor 1  │  │  Executor 2  │  │  Executor 3  │          │
│  │  (Worker A)  │  │  (Worker B)  │  │  (Worker C)  │          │
│  │              │  │              │  │              │          │
│  │  ┌────────┐  │  │  ┌────────┐  │  │  ┌────────┐  │          │
│  │  │ Task 1 │  │  │  │ Task 3 │  │  │  │ Task 5 │  │          │
│  │  └────────┘  │  │  └────────┘  │  │  └────────┘  │          │
│  │  ┌────────┐  │  │  ┌────────┐  │  │  ┌────────┐  │          │
│  │  │ Task 2 │  │  │  │ Task 4 │  │  │  │ Task 6 │  │          │
│  │  └────────┘  │  │  └────────┘  │  │  └────────┘  │          │
│  │              │  │              │  │              │          │
│  │  [Cache/     │  │  [Cache/     │  │  [Cache/     │          │
│  │   Memory]    │  │   Memory]    │  │   Memory]    │          │
│  └──────────────┘  └──────────────┘  └──────────────┘          │
└─────────────────────────────────────────────────────────────────┘
```

### 层级关系（从大到小）
```
Application
  └── Job (一个 Action = 一个 Job)
        └── Stage (以 Shuffle 为边界切分)
              └── Task (一个分区 = 一个 Task)
```

### 关键点
- **Task 数量 = 分区数量**：如果 RDD 有 200 个分区，Stage 就有 200 个 Task
- **并行度 = min(Executor cores, Task数量)**：并行运行的 Task 受 Executor 核数限制
- **Driver 是单点**：Driver 挂了整个 Application 挂了

In [ ]:
# 用 Python 模拟 Spark 应用的层级结构
# 这段代码展示了 Application -> Job -> Stage -> Task 的关系

from dataclasses import dataclass, field
from typing import List, Optional
from enum import Enum

class TaskStatus(Enum):
    PENDING = "PENDING"
    RUNNING = "RUNNING"
    SUCCESS = "SUCCESS"
    FAILED = "FAILED"

@dataclass
class Task:
    task_id: int
    partition_id: int
    executor_id: int
    status: TaskStatus = TaskStatus.PENDING
    
    def __repr__(self):
        return f"Task(id={self.task_id}, partition={self.partition_id}, executor={self.executor_id}, {self.status.value})"

@dataclass
class Stage:
    stage_id: int
    stage_type: str  # 'ShuffleMap' or 'Result'
    num_partitions: int
    tasks: List[Task] = field(default_factory=list)
    
    def create_tasks(self, executors: List[int]):
        """为每个分区创建一个 Task，按 round-robin 分配到 Executor"""
        self.tasks = [
            Task(
                task_id=self.stage_id * 100 + i,
                partition_id=i,
                executor_id=executors[i % len(executors)]
            )
            for i in range(self.num_partitions)
        ]
        return self.tasks

@dataclass  
class Job:
    job_id: int
    action: str  # 触发这个 Job 的 Action 名称
    stages: List[Stage] = field(default_factory=list)

# 模拟一个简单的 Spark 作业
# 例：df.groupBy('key').count().collect() 会触发:
#   Stage 0 (ShuffleMap): map 阶段，准备 shuffle 数据
#   Stage 1 (Result):     reduce 阶段，聚合结果

executors = [1, 2, 3]  # 3 个 Executor

stage0 = Stage(stage_id=0, stage_type="ShuffleMap", num_partitions=6)
stage1 = Stage(stage_id=1, stage_type="Result", num_partitions=3)

stage0_tasks = stage0.create_tasks(executors)
stage1_tasks = stage1.create_tasks(executors)

job = Job(job_id=0, action="collect", stages=[stage0, stage1])

print("=" * 60)
print(f"Job {job.job_id} (Action: {job.action})")
print("=" * 60)
for stage in job.stages:
    print(f"\n  Stage {stage.stage_id} [{stage.stage_type}] - {stage.num_partitions} 个分区")
    for task in stage.tasks:
        print(f"    {task}")

print(f"\n总 Task 数: {sum(len(s.tasks) for s in job.stages)}")
print(f"Executor 数: {len(executors)}")
print(f"最大并行 Task 数 (假设每个 Executor 2 core): {len(executors) * 2}")

## 2. DAG Scheduler & Task Scheduler

### 两层调度器架构

```
用户代码 (transformations + action)
         │
         ▼
┌─────────────────────┐
│    DAG Scheduler    │  ← 高层调度器
│                     │
│  1. 构建 RDD DAG    │
│  2. 识别 Stage 边界  │  (宽依赖处切割)
│  3. 提交 Stage      │
│  4. 追踪 Stage 状态  │
└─────────┬───────────┘
          │ TaskSet (一组 Task)
          ▼
┌─────────────────────┐
│   Task Scheduler    │  ← 低层调度器
│                     │
│  1. 资源分配        │
│  2. 数据本地性优化  │  (PROCESS_LOCAL > NODE_LOCAL > RACK_LOCAL > ANY)
│  3. Task 重试       │
│  4. 心跳监控        │
└─────────┬───────────┘
          │
          ▼
    Executor 集群
```

### 数据本地性优先级

| 级别 | 含义 | 性能 |
|------|------|------|
| PROCESS_LOCAL | 数据在同一 JVM 进程（缓存） | 最优 |
| NODE_LOCAL | 数据在同一节点（本地磁盘） | 好 |
| RACK_LOCAL | 数据在同一机架 | 一般 |
| ANY | 需要跨网络传输 | 最差 |

### DAG 调度关键流程
1. Action 被调用 → DAGScheduler.runJob()
2. DAGScheduler 从最终 RDD 反向遍历依赖图
3. 遇到**宽依赖**就切割 Stage（ShuffleMapStage）
4. 生成 Stage DAG，按拓扑顺序提交
5. 每个 Stage 完成后，触发下游 Stage

In [ ]:
# 模拟 DAG Scheduler 的 Stage 划分逻辑
# 展示 Spark 如何识别宽依赖并切割 Stage

from typing import Dict, Set, Tuple

class DependencyType(Enum):
    NARROW = "Narrow"   # 窄依赖：一对一，不需要 Shuffle
    WIDE = "Wide"       # 宽依赖：多对多，需要 Shuffle

@dataclass
class RDDNode:
    name: str
    operation: str
    dep_type: Optional[DependencyType] = None  # 与父节点的依赖类型
    parents: List['RDDNode'] = field(default_factory=list)
    
    def __repr__(self):
        dep_str = f"[{self.dep_type.value}]" if self.dep_type else "[Root]"
        return f"{self.name}({self.operation}) {dep_str}"

class SimpleDAGScheduler:
    """简化版 DAGScheduler - 演示 Stage 划分逻辑"""
    
    def compute_stages(self, final_rdd: RDDNode) -> Dict[int, List[RDDNode]]:
        """从最终 RDD 反向遍历，遇到宽依赖就切割 Stage"""
        stages = {}
        current_stage = 0
        
        def traverse(node: RDDNode, stage_id: int):
            if stage_id not in stages:
                stages[stage_id] = []
            stages[stage_id].append(node)
            
            for parent in node.parents:
                if parent.dep_type == DependencyType.WIDE:
                    # 宽依赖 = Stage 边界，父节点在新 Stage
                    traverse(parent, stage_id + 1)
                else:
                    # 窄依赖 = 同一 Stage
                    traverse(parent, stage_id)
        
        traverse(final_rdd, current_stage)
        return stages

# 构建 DAG：textFile -> filter -> map -> groupByKey -> map -> collect
#
#  textFile ──[窄]──► filter ──[窄]──► map ──[宽:Shuffle]──► groupByKey ──[窄]──► map
#
#  Stage 1: textFile, filter, map (到 shuffle 之前)
#  Stage 0: groupByKey, map (shuffle 之后)

text_file = RDDNode("textFile", "HadoopRDD", dep_type=None)
filtered = RDDNode("filtered", "filter", dep_type=DependencyType.NARROW, parents=[text_file])
mapped = RDDNode("mapped", "map", dep_type=DependencyType.NARROW, parents=[filtered])
grouped = RDDNode("grouped", "groupByKey", dep_type=DependencyType.WIDE, parents=[mapped])
result = RDDNode("result", "map", dep_type=DependencyType.NARROW, parents=[grouped])

scheduler = SimpleDAGScheduler()
stages = scheduler.compute_stages(result)

print("=" * 55)
print("DAG Scheduler - Stage 划分结果")
print("=" * 55)
print("\n原始 DAG:")
print("  textFile ─[窄]─► filter ─[窄]─► map ─[宽/Shuffle]─► groupByKey ─[窄]─► map")
print()
# 注意：Stage 编号是反向的（从 0 开始，0 是最终 Stage）
for stage_id in sorted(stages.keys(), reverse=True):
    nodes = stages[stage_id]
    exec_order = len(stages) - 1 - stage_id  # 实际执行顺序
    print(f"  Stage {stage_id} (执行顺序 #{exec_order}):")
    for node in reversed(nodes):  # 反向显示执行顺序
        dep_info = f" ← {node.dep_type.value}依赖" if node.dep_type else " ← 数据源"
        print(f"    [{node.operation:12}] {node.name}{dep_info}")
    print()

print("关键结论: 宽依赖 (groupByKey) 是 Stage 边界，需要 Shuffle")

## 3. 宽依赖 vs 窄依赖 & Shuffle

### 定义

**窄依赖 (Narrow Dependency)**：父 RDD 的每个分区**至多**被子 RDD 的**一个**分区使用
- 可以在单节点上流水线执行，无需网络传输
- 故障恢复快：只需重算父分区，不影响其他分区
- 典型操作：`map`, `filter`, `flatMap`, `union`, `mapPartitions`

**宽依赖 (Wide Dependency / Shuffle Dependency)**：父 RDD 的每个分区可能被子 RDD 的**多个**分区使用
- 必须等所有父 Stage 完成后才能开始（Shuffle 屏障）
- 故障恢复慢：可能需要重算所有父分区
- 典型操作：`groupByKey`, `reduceByKey`, `join`, `sortBy`, `repartition`

### 数据流对比

```
窄依赖 (Narrow) - 无 Shuffle:
┌──────────┐     ┌──────────┐
│ 父分区 0  │────►│ 子分区 0  │  一对一映射
│ [A, B, C]│     │ [a, b, c]│  (map/filter)
└──────────┘     └──────────┘
┌──────────┐     ┌──────────┐
│ 父分区 1  │────►│ 子分区 1  │
│ [D, E, F]│     │ [d, e, f]│
└──────────┘     └──────────┘

宽依赖 (Wide) - 有 Shuffle:
┌──────────┐     网络传输     ┌──────────┐
│ 父分区 0  │──────┬──────────►│ 子分区 0  │
│[a:1,b:2] │      │           │(a:1,a:3) │  按 Key 聚合
└──────────┘      │           └──────────┘  (groupByKey)
┌──────────┐      │           ┌──────────┐
│ 父分区 1  │──────┴──────────►│ 子分区 1  │
│[a:3,c:4] │                  │(b:2,c:4) │
└──────────┘                  └──────────┘
```

### Shuffle 的代价
1. **磁盘 I/O**：Shuffle write 写临时文件到磁盘
2. **网络 I/O**：Shuffle read 从其他节点拉取数据
3. **序列化/反序列化**：数据在网络传输前后需要序列化
4. **排序**：Sort-based Shuffle 需要对数据排序

### Shuffle 实现：Sort-Based Shuffle
```
Shuffle Write (Map Side):           Shuffle Read (Reduce Side):
┌────────────┐                      ┌────────────┐
│  Map Task  │                      │Reduce Task │
│  1.计算结果 │                      │  1.拉取数据 │
│  2.按分区排序│──── 磁盘文件 ────────►│  2.归并排序 │
│  3.写入文件 │    (shuffle_0_0.data)│  3.聚合计算 │
└────────────┘                      └────────────┘
```

In [ ]:
# 用 Python 模拟 Shuffle 过程
# 展示 groupByKey 的 shuffle write 和 shuffle read 阶段

import hashlib
from collections import defaultdict

def hash_partition(key: str, num_partitions: int) -> int:
    """模拟 Spark 的 hash partitioner"""
    return hash(key) % num_partitions

# 原始数据：2 个 Map 分区
map_partition_0 = [("apple", 1), ("banana", 2), ("apple", 3), ("cherry", 1)]
map_partition_1 = [("banana", 1), ("apple", 2), ("date", 5), ("cherry", 3)]

all_partitions = [map_partition_0, map_partition_1]
num_reduce_partitions = 3

print("=" * 55)
print("Shuffle 过程模拟")
print("=" * 55)

# === SHUFFLE WRITE (Map Side) ===
print("\n[Shuffle Write] Map 阶段：按目标分区写出数据")
shuffle_files = defaultdict(list)  # shuffle_files[reduce_partition_id] = [(key, value)]

for map_id, partition in enumerate(all_partitions):
    print(f"\n  Map Task {map_id} 的输出:")
    for key, value in partition:
        target_partition = hash_partition(key, num_reduce_partitions)
        shuffle_files[target_partition].append((key, value))
        print(f"    ({key!r:8}, {value}) → Reduce分区 {target_partition}")

# === SHUFFLE READ (Reduce Side) ===
print("\n" + "-" * 55)
print("\n[Shuffle Read] Reduce 阶段：拉取并聚合数据")
print(f"  (等待所有 Map Task 完成后才能开始 - 这就是 Shuffle 屏障!)")
print()

for reduce_id in range(num_reduce_partitions):
    data = shuffle_files[reduce_id]
    # 按 key 分组
    grouped = defaultdict(list)
    for key, value in data:
        grouped[key].append(value)
    
    print(f"  Reduce Task {reduce_id} 收到数据: {data}")
    print(f"  聚合结果 (groupByKey): {dict(grouped)}")
    print()

print("关键: 宽依赖操作 (groupByKey) 导致数据在网络间移动")
print("所有 Map Task 必须完成，Reduce Task 才能开始 (Stage 屏障)")

In [ ]:
# 用 Python 模拟 RDD 血统 (Lineage) 追踪
# Spark 用 Lineage 实现容错：不需要复制数据，只需记录计算步骤

class RDD:
    """简化版 RDD，演示 Lineage 追踪"""
    
    def __init__(self, data=None, name="parallelize", parent=None, 
                 transform_fn=None, dep_type=None):
        self._data = data
        self.name = name
        self.parent = parent
        self.transform_fn = transform_fn
        self.dep_type = dep_type  # 'narrow' or 'wide'
    
    def map(self, fn, name="map"):
        return RDD(name=name, parent=self, transform_fn=fn, dep_type="narrow")
    
    def filter(self, fn, name="filter"):
        return RDD(name=name, parent=self, transform_fn=fn, dep_type="narrow")
    
    def group_by_key(self):
        """宽依赖操作 - 需要 Shuffle"""
        def _group(data):
            result = defaultdict(list)
            for k, v in data:
                result[k].append(v)
            return list(result.items())
        return RDD(name="groupByKey [WIDE/SHUFFLE]", parent=self, 
                   transform_fn=_group, dep_type="wide")
    
    def compute(self):
        """按 Lineage 重新计算数据（用于容错恢复）"""
        if self._data is not None:
            return self._data
        parent_data = self.parent.compute()
        return self.transform_fn(parent_data)
    
    def lineage(self, depth=0) -> str:
        """展示 RDD 血统"""
        dep_icon = "🔀 " if self.dep_type == "wide" else "→ "
        prefix = "  " * depth
        line = f"{prefix}{dep_icon}{self.name}"
        if self.parent:
            return self.parent.lineage(depth + 1) + "\n" + line
        return line
    
    # Action: collect 触发实际计算
    def collect(self):
        print("[Action: collect] 触发 DAG 计算...")
        return self.compute()

# 构建 RDD 计算链（都是 lazy 的，不会立即执行）
data = [("apple", 1), ("banana", 2), ("apple", 3), 
        ("cherry", 1), ("banana", 1), ("apple", 2)]

rdd = RDD(data=data, name="parallelize")
filtered = rdd.filter(lambda x: x[1] > 0, name="filter(value > 0)")
mapped = filtered.map(lambda x: (x[0], x[1] * 2), name="map(value * 2)")
grouped = mapped.group_by_key()
result = grouped.map(lambda x: (x[0], sum(x[1])), name="map(sum values)")

print("=" * 55)
print("RDD Lineage (血统/依赖链):")
print("=" * 55)
print(result.lineage())
print()
print("[以上都是 Transformation，没有实际计算!]")
print()

# Action 触发计算
output = result.collect()
print("\n计算结果:")
for item in sorted(output):
    print(f"  {item}")
print()
print("容错优势: 如果某个分区失败，Spark 只需按 Lineage 重新计算该分区")
print("不需要像 Hadoop 那样复制数据到多个节点")

## 4. Catalyst Optimizer & Physical Plan

### Catalyst 优化器的四个阶段

```
用户代码 (SQL / DataFrame API)
         │
         ▼
┌─────────────────────────┐
│  1. Unresolved          │  解析前：只有语法树，不知道列名/类型
│     Logical Plan        │  "SELECT age FROM users"  ← age 是什么?
└───────────┬─────────────┘
            │ Analysis (查 Catalog)
            ▼
┌─────────────────────────┐
│  2. Analyzed            │  解析后：列名/类型已解析
│     Logical Plan        │  age → users.age: IntegerType
└───────────┬─────────────┘
            │ Optimization Rules (~1000 条规则)
            ▼
┌─────────────────────────┐
│  3. Optimized           │  应用规则：谓词下推、列裁剪、常量折叠...
│     Logical Plan        │  filter 下推到 scan 层，减少数据读取
└───────────┬─────────────┘
            │ Physical Planning (选择具体算法)
            ▼
┌─────────────────────────┐
│  4. Physical Plan       │  选择具体实现：BroadcastHashJoin / SortMergeJoin
│     (多个候选，选最优)   │  HashAggregation / SortAggregation
└───────────┬─────────────┘
            │ Code Generation (Tungsten)
            ▼
     字节码 (JVM Bytecode)
     直接编译为机器执行的代码
```

### 关键优化规则

| 优化规则 | 说明 | 示例 |
|---------|------|------|
| **谓词下推** (Predicate Pushdown) | 将 filter 尽量靠近数据源 | `WHERE age > 18` 在 scan 时过滤 |
| **列裁剪** (Column Pruning) | 只读取用到的列 | Parquet 列式存储效果显著 |
| **常量折叠** (Constant Folding) | 编译时计算常量 | `1 + 2` → `3` |
| **Join 重排序** (Join Reordering) | 小表先 join，减少中间结果 | 自动选择驱动表 |
| **聚合下推** | 在 join 前先聚合，减少数据量 | COUNT 先按分区计算 |

In [ ]:
# PySpark 示例：查看 Physical Plan
# 注意：此代码需要安装 PySpark 才能运行
# 下方展示的是实际输出的注释版本

PYSPARK_CODE = '''
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast

spark = SparkSession.builder.appName("CatalystDemo").getOrCreate()

# 创建示例 DataFrame
users = spark.createDataFrame([
    (1, "Alice", 25), (2, "Bob", 17), (3, "Charlie", 30)
], ["id", "name", "age"])

orders = spark.createDataFrame([
    (1, 100), (1, 200), (2, 150)
], ["user_id", "amount"])

# 查询：成年用户的总订单金额
result = (
    users.filter(col("age") >= 18)          # filter
         .join(orders, users.id == orders.user_id)  # join
         .groupBy("name")
         .agg({"amount": "sum"})
)

# 查看完整执行计划（4 个阶段）
result.explain(mode="extended")  # Spark 3.x
# 或者
result.explain(True)  # Spark 2.x
'''

print("PySpark 代码 (需要 PySpark 环境运行):")
print(PYSPARK_CODE)

print("="*60)
print("实际 explain(True) 输出示例 (已注释说明):")
print("="*60)

explain_output = """
== Parsed Logical Plan ==          ← 阶段1: 原始解析，列名未解析
'Aggregate ['name], ['name, 'sum('amount)]
+- 'Join Inner, ('id = 'user_id)
   :- 'Filter ('age >= 18)         ← filter 在 join 上面（用户写的顺序）
   :  +- 'UnresolvedRelation [users]
   +- 'UnresolvedRelation [orders]

== Analyzed Logical Plan ==        ← 阶段2: 列名/类型已解析
name: string, sum(amount): long
Aggregate [name#5], [name#5, sum(amount#9) AS sum(amount)#15L]
+- Join Inner, (id#3L = user_id#8L)
   :- Filter (age#4L >= 18)        ← age 现在是 age#4L: LongType
   :  +- LogicalRDD [id#3L, name#5, age#4L]
   +- LogicalRDD [user_id#8L, amount#9L]

== Optimized Logical Plan ==       ← 阶段3: Catalyst 优化后
Aggregate [name#5], [name#5, sum(amount#9) AS sum(amount)#15L]
+- Join Inner, (id#3L = user_id#8L)
   :- Filter (isnotnull(id#3L) AND (age#4L >= 18))  ← 自动加了 isnotnull
   :  +- Project [id#3L, name#5]   ← 列裁剪！age 不需要输出，只保留 id,name
   :     +- LogicalRDD [id#3L, name#5, age#4L]
   +- Filter isnotnull(user_id#8L) ← 自动加了 isnotnull 过滤
      +- LogicalRDD [user_id#8L, amount#9L]

== Physical Plan ==                ← 阶段4: 选择具体算法
*(2) HashAggregate(keys=[name#5], functions=[sum(amount#9)])  ← Hash聚合
+- Exchange hashpartitioning(name#5, 200)  ← Shuffle！200 个分区
   +- *(1) HashAggregate(keys=[name#5], functions=[partial_sum(amount#9)])
      +- *(1) BroadcastHashJoin [id#3L], [user_id#8L], Inner  ← 广播小表
         :- *(1) Filter ((age#4L >= 18) AND isnotnull(id#3L))
         :  +- *(1) Scan ExistingRDD [id#3L, name#5, age#4L]
         +- BroadcastExchange HashedRelationBroadcastMode([user_id#8L])
            +- *(1) Scan ExistingRDD [user_id#8L, amount#9L]
"""
print(explain_output)

In [ ]:
# 用 Python 模拟 Catalyst 的谓词下推优化效果
# 对比优化前后的数据处理量

import time
import random

# 生成模拟数据
random.seed(42)
users_data = [
    {"id": i, "name": f"user_{i}", "age": random.randint(10, 60), "city": random.choice(["Beijing", "Shanghai", "Guangzhou"])}
    for i in range(100_000)
]

print("=" * 55)
print("谓词下推 (Predicate Pushdown) 效果对比")
print("=" * 55)
print(f"总数据量: {len(users_data):,} 条")

# 方法1: 未优化 - 先读所有列，再过滤（没有谓词下推）
start = time.time()
result_no_opt = []
rows_scanned = 0
for row in users_data:
    rows_scanned += 1  # 扫描了完整的 row（包括所有列）
    if row["age"] >= 18 and row["city"] == "Beijing":
        result_no_opt.append({"name": row["name"]})  # 只需要 name
time_no_opt = time.time() - start

print(f"\n[未优化] 无谓词下推:")
print(f"  扫描行数: {rows_scanned:,}")
print(f"  结果行数: {len(result_no_opt):,}")
print(f"  耗时: {time_no_opt*1000:.2f} ms")

# 方法2: 优化后 - 谓词下推（先过滤，再选列）
# 在 Parquet 中，这意味着直接跳过不满足条件的 row group
# 这里用早期退出来模拟效果
start = time.time()
result_opt = []
filter_pushdown_count = 0
for row in users_data:
    # 谓词下推: 先检查过滤条件（只读必要的列）
    if row["age"] < 18 or row["city"] != "Beijing":  # 下推的过滤条件
        filter_pushdown_count += 1
        continue  # 跳过，不读取其他列
    # 只有满足条件的行才投影到需要的列（列裁剪）
    result_opt.append({"name": row["name"]})
time_opt = time.time() - start

print(f"\n[优化后] 谓词下推 + 列裁剪:")
print(f"  过滤掉的行: {filter_pushdown_count:,} ({filter_pushdown_count/len(users_data)*100:.1f}%)")
print(f"  结果行数: {len(result_opt):,}")
print(f"  耗时: {time_opt*1000:.2f} ms")

print(f"\n结论: Catalyst 通过谓词下推，在 Parquet 中可减少 {filter_pushdown_count/len(users_data)*100:.0f}% 的数据扫描")
print("实际效果在列式存储 (Parquet/ORC) 中更显著，可跳过整个 Row Group")

## 5. Tungsten 内存管理 & Code Generation

### Tungsten 项目的三大核心优化

#### 1. 内存管理 (Off-Heap Memory)
```
传统 JVM 内存:                    Tungsten Off-Heap 内存:
┌────────────────────┐            ┌────────────────────┐
│  JVM Heap          │            │  JVM Heap (小)      │
│  ┌──────────────┐  │            │  (存 Java 对象引用)  │
│  │ Java 对象     │  │            └────────────────────┘
│  │ (对象头 16B)  │  │            ┌────────────────────┐
│  │ (GC 压力大)   │  │            │  Off-Heap 内存      │
│  └──────────────┘  │            │  (紧凑二进制格式)    │
│  GC 频繁 Stop-the-World │       │  (无 GC 压力)        │
└────────────────────┘            │  (CPU 缓存友好)      │
                                  └────────────────────┘

示例：存储字符串 "hello" (5字节)
Java String 对象: ~56 bytes (11x 放大!)
Tungsten 二进制: 5 bytes
```

#### 2. Cache-Aware 计算
- 数据连续存储在内存，充分利用 CPU L1/L2 缓存
- Sort 操作直接在 8-byte 指针上操作，减少 cache miss

#### 3. Whole-Stage Code Generation (WSCG)
```
传统 Volcano 模型 (逐行, 虚函数调用):        WSCG (整体编译):

Agg.next()                                  一个大循环，无虚函数调用:
  └─ Join.next()                            for (row in inputData) {
       └─ Filter.next()                         if (row.age >= 18) {
            └─ Scan.next()                          long v = row.salary * 1.1;
                                                    agg.update(v);
每一行调用 3-4 次虚函数                         }
→ CPU 无法预测 (分支预测失败)               }  ← 编译为 JVM 字节码
→ 无法向量化                               → CPU 分支预测友好
                                           → JIT 可以进一步优化
```

### 内存区域划分

| 区域 | 用途 | 大小配置 |
|------|------|----------|
| Execution Memory | Shuffle, Sort, Join | `spark.memory.fraction` × heap |
| Storage Memory | Cache/Persist | 与 Execution 共享，动态调整 |
| User Memory | 用户数据结构 | 剩余部分 |
| Reserved Memory | 系统预留 | 固定 300MB |

**统一内存管理** (Spark 1.6+)：Execution 和 Storage 共享内存池，动态借用，互相驱逐。

In [ ]:
# 演示 Tungsten 二进制格式 vs Java 对象的内存效率
import sys
import struct

print("=" * 55)
print("Tungsten 内存效率演示")
print("=" * 55)

# Java 对象的内存开销 (用 Python 近似模拟)
class JavaLikeRow:
    """模拟 Java Row 对象的内存开销"""
    def __init__(self, name: str, age: int, salary: float):
        self.name = name      # String 对象：字符数据 + 对象头 + 引用
        self.age = age        # Integer 对象（装箱）：值 + 对象头
        self.salary = salary  # Double 对象（装箱）：值 + 对象头

# 创建示例数据
rows = [
    JavaLikeRow("Alice", 25, 80000.0),
    JavaLikeRow("Bob", 30, 95000.0),
    JavaLikeRow("Charlie", 28, 72000.0),
]

# Python 对象大小（近似 Java 开销）
python_sizes = []
for row in rows:
    size = sys.getsizeof(row) + sys.getsizeof(row.name) + sys.getsizeof(row.age) + sys.getsizeof(row.salary)
    python_sizes.append(size)

print("\n[Java/Python 对象方式]")
for i, (row, size) in enumerate(zip(rows, python_sizes)):
    print(f"  Row {i}: name='{row.name}', age={row.age}, salary={row.salary:.0f}")
    print(f"    内存占用: ~{size} bytes (含对象头、GC元数据等开销)")
total_obj = sum(python_sizes)
print(f"  总计: {total_obj} bytes")

# Tungsten 二进制格式 (紧凑存储)
print("\n[Tungsten 二进制格式]")
print("  格式: [null bitmap(8B)] [fixed-length fields] [variable-length data]")

def tungsten_pack(name: str, age: int, salary: float) -> bytes:
    """模拟 Tungsten 的 UnsafeRow 格式"""
    name_bytes = name.encode('utf-8')
    name_len = len(name_bytes)
    # null bitmap (8 bytes) + age(8B long) + salary(8B double) + name offset+len(8B) + name data
    return struct.pack(
        f'QqddQ{name_len}s',
        0,           # null bitmap (8 bytes, no nulls)
        age,         # age as long (8 bytes)
        salary,      # salary as double (8 bytes)
        name_len,    # offset to name (8 bytes)
        name_bytes   # name data
    )

tungsten_sizes = []
for row in rows:
    packed = tungsten_pack(row.name, row.age, row.salary)
    tungsten_sizes.append(len(packed))
    print(f"  Row: name='{row.name}', age={row.age}, salary={row.salary:.0f}")
    print(f"    内存占用: {len(packed)} bytes (紧凑二进制)")
    print(f"    十六进制: {packed.hex()}")

total_tungsten = sum(tungsten_sizes)
print(f"  总计: {total_tungsten} bytes")

print(f"\n内存节省: {total_obj - total_tungsten} bytes ({(1 - total_tungsten/total_obj)*100:.1f}%)")
print("更重要的优势:")
print("  1. 连续内存存储 → CPU 缓存命中率高")
print("  2. 不在 JVM Heap → 无 GC 压力")
print("  3. 可直接用 sun.misc.Unsafe 操作 → 比 Java 对象访问快")

In [ ]:
# 演示 Whole-Stage Code Generation 的概念
# 对比 Volcano 模型 vs WSCG 的性能差异

import time

data = list(range(1_000_000))  # 100万条数据

# 模拟 Volcano 模型：每个算子独立调用 next()
# 相当于多个 Python 生成器串联（有函数调用开销）
def scan_op(data):
    """Scan 算子"""
    for x in data:
        yield x

def filter_op(source, threshold):
    """Filter 算子：age >= threshold"""
    for x in source:
        if x >= threshold:
            yield x

def project_op(source, multiplier):
    """Project 算子：salary * multiplier"""
    for x in source:
        yield x * multiplier

# Volcano 模型：链式迭代器
start = time.time()
total_volcano = 0
count_volcano = 0
for val in project_op(filter_op(scan_op(data), 500_000), 2):
    total_volcano += val
    count_volcano += 1
time_volcano = time.time() - start

# WSCG 模拟：一个融合的大循环（无函数调用开销）
start = time.time()
total_wscg = 0
count_wscg = 0
threshold = 500_000
multiplier = 2
for x in data:  # 一个循环，无函数调用
    if x >= threshold:  # filter 内联
        total_wscg += x * multiplier  # project 内联
        count_wscg += 1
time_wscg = time.time() - start

print("=" * 55)
print("Whole-Stage Code Generation 效果对比")
print("=" * 55)
print(f"数据量: {len(data):,} 条")
print(f"\n[Volcano 模型] 链式迭代器 (虚函数调用):")
print(f"  结果: count={count_volcano:,}, sum={total_volcano:,}")
print(f"  耗时: {time_volcano*1000:.2f} ms")
print(f"\n[WSCG 模拟] 融合单循环 (无虚函数调用):")
print(f"  结果: count={count_wscg:,}, sum={total_wscg:,}")
print(f"  耗时: {time_wscg*1000:.2f} ms")
print(f"\n加速比: {time_volcano/time_wscg:.2f}x")
print()
print("在真实 JVM 中，WSCG 的优势更显著：")
print("  - 消除虚函数调用 (多态分发)") 
print("  - 允许 JIT 编译器内联代码")
print("  - 数据保留在 CPU 寄存器中（不写回内存）")
print("  - 真实场景可提速 2-10x")

## 复习要点

### 高频考点速记

| 概念 | 关键记忆点 |
|------|------------|
| **Application** | 一个 SparkContext 的生命周期 |
| **Job** | 一个 Action 触发一个 Job |
| **Stage** | 以宽依赖(Shuffle)为边界 |
| **Task** | 最小执行单元，一个分区对应一个 Task |
| **DAGScheduler** | 划分 Stage，构建 Stage DAG |
| **TaskScheduler** | 分配资源，考虑数据本地性 |
| **窄依赖** | 一对一，无 Shuffle，可流水线 |
| **宽依赖** | 多对多，需 Shuffle，Stage 边界 |
| **Catalyst** | 4阶段：解析→分析→优化→物理计划 |
| **Tungsten** | 二进制格式 + 堆外内存 + WSCG |

### 常见面试问题
1. Task 的数量由什么决定？→ 分区数量
2. 什么触发 Stage 划分？→ 宽依赖（Shuffle）
3. Catalyst 的谓词下推是什么？→ Filter 下推到数据源，减少读取量
4. Tungsten 为什么比普通 JVM 快？→ 堆外内存(无GC) + 紧凑二进制 + WSCG
5. 数据本地性优先级？→ PROCESS_LOCAL > NODE_LOCAL > RACK_LOCAL > ANY

---

## 练习题

---

### 练习 1：Stage 划分分析

给定以下 PySpark 代码，请分析会产生几个 Stage，每个 Stage 包含哪些操作？

```python
rdd = sc.textFile("data.txt")           # 分区数: 100
words = rdd.flatMap(lambda x: x.split())
pairs = words.map(lambda w: (w, 1))
counts = pairs.reduceByKey(lambda a, b: a + b)  # 200 partitions
filtered = counts.filter(lambda x: x[1] > 5)
sorted_counts = filtered.sortByKey()    # 200 partitions
result = sorted_counts.collect()        # Action
```

In [ ]:
# 练习 1 答案

answer_1 = """
答案：共 3 个 Stage

Stage 0 (ShuffleMap):
  操作: textFile → flatMap → map → [Shuffle for reduceByKey]
  Task 数: 100 (原始分区数)
  边界: reduceByKey 是宽依赖，触发 Shuffle

Stage 1 (ShuffleMap):
  操作: reduceByKey → filter → [Shuffle for sortByKey]
  Task 数: 200 (reduceByKey 默认输出分区数)
  边界: sortByKey 是宽依赖，需要全局排序，触发 Shuffle

Stage 2 (ResultStage):
  操作: sortByKey → collect
  Task 数: 200
  边界: collect 是 Action，触发整个计算

关键点:
  - reduceByKey 和 sortByKey 都是宽依赖 → 两个 Shuffle 边界
  - filter 是窄依赖，跟随 Stage 1（前一个 reduceByKey 之后）
  - 总 Task 数: 100 + 200 + 200 = 500
"""
print(answer_1)

### 练习 2：宽依赖 vs 窄依赖判断

判断以下操作是宽依赖还是窄依赖，并解释原因：

| 操作 | 宽/窄依赖 | 理由 |
|------|-----------|------|
| `map()` | ? | ? |
| `filter()` | ? | ? |
| `flatMap()` | ? | ? |
| `groupByKey()` | ? | ? |
| `reduceByKey()` | ? | ? |
| `join()` (普通) | ? | ? |
| `union()` | ? | ? |
| `coalesce()` (减少分区) | ? | ? |
| `repartition()` | ? | ? |
| `sortBy()` | ? | ? |

In [ ]:
# 练习 2 答案

dependencies = [
    ("map()",           "窄依赖", "每个父分区只对应一个子分区，一对一转换"),
    ("filter()",        "窄依赖", "过滤不改变分区关系，仍然一对一"),
    ("flatMap()",       "窄依赖", "一个输入行产生多个输出，但仍在同一分区内"),
    ("groupByKey()",    "宽依赖", "相同 key 来自不同分区，需要 Shuffle 汇集"),
    ("reduceByKey()",   "宽依赖", "需要 Shuffle，但比 groupByKey 好（在 map 端先聚合）"),
    ("join() (普通)",   "宽依赖", "两个 RDD 需要按 key shuffle 到同一分区"),
    ("union()",         "窄依赖", "只是拼接两个 RDD，各分区独立，无需重新分布"),
    ("coalesce(减少)",  "窄依赖", "减少分区时可以合并相邻分区，无需 Shuffle（但 shuffle=True 除外）"),
    ("repartition()",   "宽依赖", "内部调用 coalesce(shuffle=True)，强制 Shuffle 重分区"),
    ("sortBy()",        "宽依赖", "全局排序需要 range partition，必须 Shuffle"),
]

print(f"{'操作':<20} {'依赖类型':<10} 理由")
print("-" * 75)
for op, dep_type, reason in dependencies:
    icon = "🔀" if dep_type == "宽依赖" else "→ "
    print(f"{op:<20} {icon} {dep_type:<8} {reason}")

### 练习 3：Catalyst 优化分析

给定以下 SQL 查询，Catalyst 会应用哪些优化规则？

```sql
SELECT u.name, SUM(o.amount * 1.1) as total
FROM users u
JOIN orders o ON u.id = o.user_id
WHERE u.age >= 18
  AND u.country = 'China'
  AND o.status = 'completed'
GROUP BY u.name
HAVING SUM(o.amount * 1.1) > 100
```

users 表: 10 million 行  
orders 表: 100 million 行

In [ ]:
# 练习 3 答案

answer_3 = """
Catalyst 会应用以下优化规则:

1. 谓词下推 (Predicate Pushdown):
   - 'u.age >= 18' 和 'u.country = China' 下推到 users 表扫描
   - 'o.status = completed' 下推到 orders 表扫描
   - 效果：在 join 之前先过滤，大幅减少参与 join 的数据量

2. 列裁剪 (Column Pruning):
   - users 表只读取: id, name, age, country (不读其他列)
   - orders 表只读取: user_id, amount, status
   - 对 Parquet 格式效果显著

3. 常量折叠 (Constant Folding):
   - 1.1 是常量，但这里是字段乘以常量，无法折叠
   - 如果是 '1 + 1' 这样的纯常量表达式，会在编译期计算

4. Join 策略选择 (Physical Planning):
   - users 过滤后可能变小 → 考虑 BroadcastHashJoin
   - 如果过滤后 users 仍然大 → SortMergeJoin
   - 阈值: spark.sql.autoBroadcastJoinThreshold (默认 10MB)

5. HAVING 转换:
   - HAVING SUM(...) > 100 转换为聚合后的 Filter
   - 等价于: GROUP BY 后 filter(sum > 100)

6. 两阶段聚合 (Partial Aggregation):
   - 先在 map 端做局部 SUM（减少 Shuffle 数据量）
   - 再在 reduce 端做全局 SUM

优化后执行顺序:
  Scan(users, filter=[age>=18, country=China], cols=[id,name])
  Scan(orders, filter=[status=completed], cols=[user_id, amount])
  BroadcastHashJoin (或 SortMergeJoin)
  PartialAgg(groupBy=name, sum=amount*1.1)
  Shuffle
  FinalAgg
  Filter(sum > 100)  ← HAVING
"""
print(answer_3)

### 练习 4：故障恢复分析

在以下场景中，Spark 如何进行故障恢复？

**场景**：一个 Job 有 3 个 Stage，Stage 0 和 Stage 1 已完成，Stage 2 中的 Task 3 失败了。

```
Stage 0 (100 tasks) → Stage 1 (200 tasks) → Stage 2 (200 tasks)
                                                    ^
                                               Task 3 失败
```

In [ ]:
# 练习 4 答案

answer_4 = """
Spark 的故障恢复机制：

情况 A: Stage 2 的 Task 3 失败（但 Stage 1 Shuffle 数据还在磁盘上）
  1. Task Scheduler 检测到 Task 3 失败（心跳超时或异常）
  2. 在同一 Stage 2 重新调度 Task 3（可能分配到不同 Executor）
  3. Task 3 从 Stage 1 的 Shuffle 输出中重新读取分区 3 的数据
  4. 仅重算 Task 3，不影响其他 199 个 Task
  5. 默认最多重试 4 次 (spark.task.maxFailures)

情况 B: Stage 1 的 Shuffle 数据所在节点挂了（Shuffle 数据丢失）
  1. Stage 2 的 Task 3 无法拉取 Stage 1 的 Shuffle 数据
  2. DAGScheduler 检测到 FetchFailed 异常
  3. 需要重新提交 Stage 1（或部分 Stage 1 的 Task）
  4. Stage 0 的 Shuffle 数据如果还在 → 只重算 Stage 1
  5. Stage 0 数据也丢失 → 需要从最初数据源重算

为什么不需要复制数据就能容错？
  → RDD Lineage! Spark 记录了每步的转换逻辑
  → 只需要重新执行记录的转换，就能重建任何 RDD
  → 代价是重算时间，而不是存储冗余副本

与 Hadoop 对比:
  Hadoop: 每步结果写入 HDFS（3副本），故障恢复快但磁盘 I/O 大
  Spark: 利用 Lineage 重算，省磁盘 I/O，但恢复时有计算代价
"""
print(answer_4)

### 练习 5：架构设计题

你有一个 Spark Job 处理 10TB 数据，但发现大部分时间花在 Shuffle 上。请列出至少 5 种减少 Shuffle 的策略。

In [ ]:
# 练习 5 答案

strategies = [
    {
        "策略": "1. 使用 reduceByKey 替代 groupByKey",
        "原理": "reduceByKey 在 map 端先聚合，减少 Shuffle 传输量",
        "效果": "数据量减少 50-90%"
    },
    {
        "策略": "2. Broadcast Join 替代 Shuffle Join",
        "原理": "将小表广播到所有节点，大表无需 Shuffle",
        "效果": "完全消除一侧的 Shuffle"
    },
    {
        "策略": "3. 使用 Bucketing 预分区",
        "原理": "将常用 Join key 预先 bucket，join 时无需 Shuffle",
        "效果": "消除重复 Job 的 Shuffle"
    },
    {
        "策略": "4. 合理设置 spark.sql.shuffle.partitions",
        "原理": "分区太少 → 每个 Task 数据太多（OOM）；太多 → 小文件/调度开销",
        "效果": "优化 Shuffle 后的读取性能"
    },
    {
        "策略": "5. 开启 AQE（Adaptive Query Execution）",
        "原理": "自动合并小分区、自动切换 Join 策略",
        "效果": "自动优化，减少不必要的 Shuffle"
    },
    {
        "策略": "6. 过滤后再 Join（谓词下推）",
        "原理": "先 filter 减小数据量，再 Shuffle",
        "效果": "减少参与 Shuffle 的数据量"
    },
    {
        "策略": "7. 使用 Salting 处理数据倾斜",
        "原理": "热点 key 加随机前缀分散到多个分区",
        "效果": "避免单 Task Shuffle 数据过大"
    },
]

print("=" * 60)
print("减少 Shuffle 的 7 种策略")
print("=" * 60)
for s in strategies:
    print(f"\n{s['策略']}")
    print(f"  原理: {s['原理']}")
    print(f"  效果: {s['效果']}")